In [1]:
# %%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.5.0" {install_numpy} \
    "transformers>=4.56.0" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes

# Install evaluation libraries
!pip install -qqq nltk rouge-score python-Levenshtein scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 48.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load model using Unsloth FastLanguageModel for function calling
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Increased for function calling complexity
dtype = torch.float16
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # Changed to Llama 3 for better function calling

# Load model and tokenizer with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # Use 4-bit quantization to save memory
    full_finetuning=False,
)

print("\u2705 Model and tokenizer loaded successfully!")
print(f"Model: {model_name}")
print(f"Max sequence length: {max_seq_length}")
print(f"Using dtype: {dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model and tokenizer loaded successfully!
Model: meta-llama/Llama-3.2-3B-Instruct
Max sequence length: 2048
Using dtype: torch.float16


In [4]:
# ============================================================================
# UTILITY FUNCTIONS FOR FUNCTION CALLING FINE-TUNING WITH TOOL SUPPORT
# ============================================================================

import json
import re
import sys


def extract_json_from_response(response: str) -> dict:
    """
    Extract JSON function call from model response.
    Handles NESTED JSON objects (e.g. {"name": "func", "arguments": {"intent": "research"}}).
    """
    if not response:
        return None

    # Method 1: Try to parse the whole response as JSON
    try:
        return json.loads(response.strip())
    except:
        pass

    # Method 2: Find JSON using bracket counting (handles nested braces)
    start_idx = response.find('{')
    if start_idx == -1:
        return None

    depth = 0
    in_string = False
    escape_next = False

    for i in range(start_idx, len(response)):
        c = response[i]

        if escape_next:
            escape_next = False
            continue

        if c == '\\':
            escape_next = True
            continue

        if c == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if c == '{':
            depth += 1
        elif c == '}':
            depth -= 1
            if depth == 0:
                json_str = response[start_idx:i+1]
                try:
                    return json.loads(json_str)
                except:
                    # Try next opening brace
                    next_start = response.find('{', start_idx + 1)
                    if next_start != -1 and next_start < i:
                        start_idx = next_start
                        depth = 1  # reset
                    else:
                        return None

    return None


def validate_function_call(response: str) -> bool:
    """
    Check if response contains valid function call JSON

    Returns:
        True if valid JSON with 'name' and ('parameters' or 'arguments') fields
    """
    try:
        data = extract_json_from_response(response)
        if data and "name" in data and ("parameters" in data or "arguments" in data):
            return True
    except:
        pass
    return False


def handle_function_calling(query: str, tokenizer, model, tools=None):
    """
    Handle function calling for market research tasks with tool support
    """
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a market research analyst. You must respond with valid JSON function calls."
            },
            {
                "role": "user",
                "content": query
            }
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            model_input = tokenizer.apply_chat_template(
                messages,
                tools=tools,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            user_msgs = [m.get("content", "") for m in messages if m.get("role") == "user"]
            model_input = user_msgs[-1] if user_msgs else query

        model_inputs = tokenizer([model_input], return_tensors="pt")

        if torch.cuda.is_available():
            model_inputs = model_inputs.to('cuda')

        with torch.no_grad():
            generated_ids = model.generate(
                model_inputs.input_ids,
                attention_mask=model_inputs.attention_mask,
                max_new_tokens=500,
                temperature=0.3,
                top_p=0.95,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        json_data = extract_json_from_response(response)
        is_valid = json_data is not None and "name" in json_data and ("parameters" in json_data or "arguments" in json_data)

        return {
            "success": is_valid,
            "response": response,
            "json_data": json_data,
            "valid_function_call": is_valid
        }

    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "valid_function_call": False
        }


print("\u2705 Function calling utilities loaded successfully!")

# Quick test of JSON extraction with nested braces
test_nested = '{"name": "classify_intent", "arguments": {"intent": "research", "response": ""}}'
result = extract_json_from_response(test_nested)
print(f"   JSON extraction test (nested): {'PASS' if result and 'arguments' in result else 'FAIL'}")
test_flat = '{"name": "test", "parameters": {}}'
result2 = extract_json_from_response(test_flat)
print(f"   JSON extraction test (flat): {'PASS' if result2 and 'name' in result2 else 'FAIL'}")

✅ Function calling utilities loaded successfully!
   JSON extraction test (nested): PASS
   JSON extraction test (flat): PASS


In [5]:
# ============================================================================
# SETUP LoRA FOR FUNCTION CALLING + LOAD TOOL DEFINITIONS
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("\u2705 LoRA setup completed (optimized for function calling)!")

print("\n" + "="*70)
print("\U0001f527 LOADING TOOL DEFINITIONS FROM tool_definitions.py")
print("="*70)

import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/AI/do an')

TOOL_NAMES = [
    "INPUT_VALIDATION_TOOLS",
    "INTENT_CLASSIFICATION_TOOLS",
    "CLARIFICATION_TOOLS",
    "PLANNING_TOOLS",
    "REACT_TOOLS",
    "GENERATE_SEARCH_QUERIES_TOOLS",
    "SYNTHESIS_TOOLS",
    "CAMPAIGN_TOOLS",
    "SWOT_TOOLS",
    "CONTENT_EXPANSION_TOOLS",
]

tool_sets = {}
loaded_count = 0
failed_tools = []

print("\n\U0001f4cb Attempting to load tools:\n")

for tool_name in TOOL_NAMES:
    try:
        from tool_definitions import *
        tool_obj = eval(tool_name)
        tool_sets[tool_name] = tool_obj
        loaded_count += 1
        print(f"   \u2705 {tool_name:<35} - Loaded successfully")
    except Exception as e:
        failed_tools.append((tool_name, str(e)))
        print(f"   \u26a0\ufe0f  {tool_name:<35} - Failed: {str(e)[:50]}")

print("\n" + "="*70)
print(f"\U0001f4ca TOOL LOADING SUMMARY")
print("="*70)
print(f"\u2705 Successfully loaded: {loaded_count}/{len(TOOL_NAMES)} tools")

if failed_tools:
    print(f"\u274c Failed to load: {len(failed_tools)} tools")
    for tool_name, error in failed_tools:
        print(f"   - {tool_name}: {error}")
else:
    print("\U0001f389 All tools loaded successfully!")

print(f"\n\U0001f4da Available tools in tool_sets:")
for i, (tool_name, tool_obj) in enumerate(tool_sets.items(), 1):
    if isinstance(tool_obj, list):
        print(f"   {i}. {tool_name:<35} ({len(tool_obj)} tool(s))")
    else:
        print(f"   {i}. {tool_name:<35} (single tool)")

print("="*70 + "\n")

Unsloth 2026.5.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA setup completed (optimized for function calling)!

🔧 LOADING TOOL DEFINITIONS FROM tool_definitions.py

📋 Attempting to load tools:

   ✅ INPUT_VALIDATION_TOOLS              - Loaded successfully
   ✅ INTENT_CLASSIFICATION_TOOLS         - Loaded successfully
   ✅ CLARIFICATION_TOOLS                 - Loaded successfully
   ✅ PLANNING_TOOLS                      - Loaded successfully
   ✅ REACT_TOOLS                         - Loaded successfully
   ✅ GENERATE_SEARCH_QUERIES_TOOLS       - Loaded successfully
   ✅ SYNTHESIS_TOOLS                     - Loaded successfully
   ✅ CAMPAIGN_TOOLS                      - Loaded successfully
   ✅ SWOT_TOOLS                          - Loaded successfully
   ✅ CONTENT_EXPANSION_TOOLS             - Loaded successfully

📊 TOOL LOADING SUMMARY
✅ Successfully loaded: 10/10 tools
🎉 All tools loaded successfully!

📚 Available tools in tool_sets:
   1. INPUT_VALIDATION_TOOLS              (1 tool(s))
   2. INTENT_CLASSIFICATION_TOOLS         (1 tool(s

In [ ]:
# ============================================================================
# TRAINING DATA STRUCTURE EXPLANATION WITH SAMPLE
# ============================================================================

print("="*90)
print("📊 TRAINING DATA STRUCTURE: Complete Format with Tools")
print("="*90)

sample_data_with_tools = {
    "messages": [
        {
            "role": "system",
            "content": "As an experienced market intelligence specialist, your role is to understand what users are asking for and respond conversationally."
        },
        {
            "role": "user",
            "content": "Giáo dục online ở Việt Nam hiệu quả như thế nào so với các nước khác? Dữ liệu mới nhất là gì?"
        },
        {
            "role": "tool",
            "tool_calls": "INTENT_CLASSIFICATION_TOOLS"
        },
        {
            "role": "assistant",
            "content": "{\"name\": \"classify_intent\", \"arguments\": {\"intent\": \"research\", \"response\": \"\", \"reasoning\": \"User asks for comparative analysis of online education effectiveness in Vietnam versus other countries and requests the latest data, which requires market research and external information gathering.\"}}"
        }
    ]
}

print("\n📋 COMPLETE DATA SAMPLE (JSON):\n")
print(json.dumps(sample_data_with_tools, indent=2, ensure_ascii=False))

print("\n" + "="*90)
print("🔍 DATA STRUCTURE BREAKDOWN:")
print("="*90)

print("""
1️⃣ SYSTEM MESSAGE:
   - Role: "system"
   - Content: Instruction về cách model nên hành động (diverse system prompts)
   - Ví dụ: "As an experienced market intelligence specialist..."
   
2️⃣ USER MESSAGE:
   - Role: "user"
   - Content: Câu hỏi/yêu cầu từ người dùng
   - Format: Tiếng Việt, natural language
   
3️⃣ TOOL MESSAGE (CẦN THIẾT):
   - Role: "tool"
   - tool_calls: Tên của tool set được sử dụng (VD: "INTENT_CLASSIFICATION_TOOLS")
   - Purpose: Cho model biết tool nào sẽ được sử dụng
   - Available tools:
     • INTENT_CLASSIFICATION_TOOLS (classify_intent)
     • INPUT_VALIDATION_TOOLS (validate_input)
     • CLARIFICATION_TOOLS (generate_questions)
     • PLANNING_TOOLS (create_research_plan)
     • REACT_TOOLS (decide_action)
     • GENERATE_SEARCH_QUERIES_TOOLS (generate_search_queries)
     • SYNTHESIS_TOOLS (synthesize_overview)
     • CAMPAIGN_TOOLS (create_campaign_plan)
     • SWOT_TOOLS (generate_swot)
     • CONTENT_EXPANSION_TOOLS (expand_content)

4️⃣ ASSISTANT MESSAGE:
   - Role: "assistant"
   - Content: Valid JSON response với function call
   - Format: {"name": "function_name", "arguments": {...}} hoặc "parameters"
   - Required fields:
     • "name": Tên function được gọi
     • "arguments" hoặc "parameters": Chi tiết arguments
     • "intent": Loại intent (research/chat/knowledge)
     • "reasoning": (Optional) Giải thích quyết định
""")

print("="*90)
print("✅ NGUYÊN TẮC:")
print("="*90)
print("""
✓ Mỗi training example phải có 4 messages: system → user → tool → assistant
✓ Tool message PHẢI match với tool thực tế có trong tool_definitions.py
✓ Assistant response phải là valid JSON
✓ System prompt nên diverse (7+ phiên bản khác nhau)
✓ User input phải tiếng Việt hoặc English
✓ Intent phải là một trong: "research", "chat", "knowledge"
✓ Response field có thể empty string "" nếu intent = "research" hoặc "knowledge"
""")

print("\n" + "="*90)
print("🧪 SAMPLE TRAINING EXAMPLES (28 hiện tại):")
print("="*90)

# Hiển thị 3 ví dụ từ training_data.json
if len(raw_training_data) >= 3:
    for i in range(min(3, len(raw_training_data))):
        example = raw_training_data[i]
        print(f"\n📌 EXAMPLE {i+1}:")
        print("-" * 90)
        
        # Extract info
        system_msg = ""
        user_msg = ""
        tool_msg = ""
        assistant_msg = ""
        
        for msg in example.get("messages", []):
            if msg.get("role") == "system":
                system_msg = msg.get("content", "")[:80]
            elif msg.get("role") == "user":
                user_msg = msg.get("content", "")[:100]
            elif msg.get("role") == "tool":
                tool_msg = msg.get("tool_calls", "")
            elif msg.get("role") == "assistant":
                assistant_msg = msg.get("content", "")[:150]
        
        print(f"   System:    {system_msg}...")
        print(f"   User:      {user_msg}...")
        print(f"   Tool:      {tool_msg}")
        print(f"   Assistant: {assistant_msg}...")

print("\n" + "="*90)


In [6]:
# ============================================================================
# LOAD TRAINING DATA + SPLIT 90/10 FOR TRAIN/VALIDATION
# ============================================================================

import json
import random
from collections import defaultdict
from datasets import Dataset

print("\U0001f4e5 Loading training data...")

with open("/content/drive/MyDrive/Colab Notebooks/AI/do an/training_data.json", "r", encoding="utf-8") as f:
    raw_training_data = json.load(f)

print(f"\u2705 Loaded {len(raw_training_data)} total examples from training_data.json")

# Convert to messages format with tool set mapping (for ALL data)
all_examples = []
for example_idx, example in enumerate(raw_training_data):
    messages = None
    if "messages" in example:
        messages = example.get("messages", [])
    elif "conversation" in example:
        messages = example.get("conversation", [])

    if messages and len(messages) > 0:
        tools_for_this_example = None
        tool_calls_name = None

        filtered_messages = []
        for msg in messages:
            if msg.get("role") == "tool":
                tool_calls_name = msg.get("tool_calls", "")
                if tool_calls_name in tool_sets:
                    tools_for_this_example = tool_sets[tool_calls_name]
            else:
                filtered_messages.append(msg)

        # Extract expected intent for stratified splitting
        expected_intent = None
        for msg in messages:
            if msg.get("role") == "assistant":
                content = msg.get("content", "")
                try:
                    json_data = json.loads(content)
                    expected_intent = json_data.get("arguments", {}).get("intent")
                    if not expected_intent:
                        expected_intent = json_data.get("parameters", {}).get("intent")
                except:
                    pass

        all_examples.append({
            "messages": filtered_messages,
            "tools": tools_for_this_example,
            "tool_calls_name": tool_calls_name,
            "expected_intent": expected_intent,
            "raw_messages": messages,
        })

print(f"\u2705 Converted {len(all_examples)} examples with tool mappings")

# ============================================================================
# STRATIFIED SPLIT: 90% TRAIN / 10% VALIDATION
# ============================================================================

print("\n" + "="*70)
print("\U0001f4ca SPLITTING DATA: 90% TRAINING / 10% VALIDATION")
print("="*70)

random.seed(3407)

intent_groups = defaultdict(list)
for idx, ex in enumerate(all_examples):
    intent = ex.get("expected_intent", "unknown") or "unknown"
    intent_groups[intent].append(idx)

train_indices = []
val_indices = []

for intent, indices in intent_groups.items():
    random.shuffle(indices)
    n_val = max(1, int(len(indices) * 0.1))
    val_indices.extend(indices[:n_val])
    train_indices.extend(indices[n_val:])

training_examples = [all_examples[i] for i in train_indices]
validation_raw = [all_examples[i] for i in val_indices]

# Parse validation examples into evaluation format
validation_examples = []
for example in validation_raw:
    raw_messages = example["raw_messages"]
    system_msg = None
    user_msg = None
    assistant_msg = None
    expected_intent = None
    expected_response = None

    for msg in raw_messages:
        role = msg.get("role")
        if role == "system":
            system_msg = msg.get("content", "")
        elif role == "user":
            user_msg = msg.get("content", "")
        elif role == "assistant":
            content = msg.get("content", "")
            assistant_msg = content
            try:
                json_data = json.loads(content)
                expected_intent = json_data.get("arguments", {}).get("intent")
                expected_response = json_data.get("arguments", {}).get("response", "")
                if not expected_intent:
                    expected_intent = json_data.get("parameters", {}).get("intent")
                    expected_response = json_data.get("parameters", {}).get("response", "")
            except:
                pass

    if user_msg and assistant_msg:
        validation_examples.append({
            "user_input": user_msg,
            "full_messages": raw_messages,
            "assistant_response": assistant_msg,
            "expected_intent": expected_intent,
            "expected_response": expected_response,
            "system_prompt": system_msg,
            "tools": example["tools"],
            "tool_calls_name": example.get("tool_calls_name"),
        })

print(f"\n\U0001f4ca Split Results:")
print(f"   \u2022 Total examples:      {len(all_examples)}")
print(f"   \u2022 Training examples:    {len(training_examples)} ({len(training_examples)*100//len(all_examples)}%)")
print(f"   \u2022 Validation examples:  {len(validation_examples)} ({len(validation_examples)*100//len(all_examples)}%)")

print(f"\n\U0001f4ca Intent distribution (Train / Validation):")
train_intents = defaultdict(int)
val_intents = defaultdict(int)
for ex in training_examples:
    intent = ex.get("expected_intent", "unknown") or "unknown"
    train_intents[intent] += 1
for ex in validation_examples:
    intent = ex.get("expected_intent", "unknown") or "unknown"
    val_intents[intent] += 1

all_intent_keys = sorted(set(list(train_intents.keys()) + list(val_intents.keys())))
print(f"   {'Intent':<25} {'Train':>8} {'Val':>8}")
print(f"   {'\u2500'*25} {'\u2500'*8} {'\u2500'*8}")
for intent in all_intent_keys:
    print(f"   {intent:<25} {train_intents.get(intent, 0):>8} {val_intents.get(intent, 0):>8}")

print(f"\n{'='*70}")

📥 Loading training data...
✅ Loaded 299 total examples from training_data.json
✅ Converted 299 examples with tool mappings

📊 SPLITTING DATA: 90% TRAINING / 10% VALIDATION

📊 Split Results:
   • Total examples:      299
   • Training examples:    271 (90%)
   • Validation examples:  28 (9%)

📊 Intent distribution (Train / Validation):
   Intent                       Train      Val
   ───────────────────────── ──────── ────────
   chat                            36        4
   knowledge                       47        5
   research                        54        5
   unknown                        134       14



In [7]:
# ============================================================================
# FORMAT TRAINING DATASET + TRAINING CONFIG
# ============================================================================

from datasets import Dataset

def format_function_calling(idx):
    messages = training_examples[idx]['messages']
    example_tools = training_examples[idx]['tools']

    if hasattr(tokenizer, "apply_chat_template"):
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tools=example_tools if example_tools else None,
                tokenize=False,
                add_generation_prompt=False
            )
        except Exception as e:
            print(f"   \u26a0\ufe0f Error formatting example {idx}: {e}")
            try:
                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False
                )
            except Exception as e2:
                print(f"   \u274c Fallback also failed: {e2}")
                text = ""
                for msg in messages:
                    role = msg.get('role', 'unknown')
                    content = msg.get('content', '')
                    text += f"<{role}>{content}</{role}>\n"
    else:
        text = ""
        for msg in messages:
            role = msg.get('role', 'unknown')
            content = msg.get('content', '')
            text += f"<{role}>{content}</{role}>\n"
    return text

print("\U0001f504 Formatting training dataset with per-example tools...")
texts = [format_function_calling(i) for i in range(len(training_examples))]
dataset = Dataset.from_dict({"text": texts})

print(f"\u2705 Training dataset formatted: {len(dataset)} examples")
print(f"Sample text length: {len(dataset[0]['text'])} characters")
print(f"First 300 chars:\n{dataset[0]['text'][:300]}...")

# Format validation dataset for eval during training
print("\n\U0001f504 Formatting validation dataset for eval during training...")

def format_val_for_training(idx):
    raw = validation_raw[idx]
    messages = raw['messages']
    example_tools = raw['tools']
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            text = tokenizer.apply_chat_template(
                messages, tools=example_tools if example_tools else None,
                tokenize=False, add_generation_prompt=False
            )
        except:
            try:
                text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False
                )
            except:
                text = ""
                for msg in messages:
                    text += f"<{msg.get('role','unknown')}>{msg.get('content','')}</{msg.get('role','unknown')}>\n"
    else:
        text = ""
        for msg in messages:
            text += f"<{msg.get('role','unknown')}>{msg.get('content','')}</{msg.get('role','unknown')}>\n"
    return text

val_texts = [format_val_for_training(i) for i in range(len(validation_raw))]
eval_dataset = Dataset.from_dict({"text": val_texts})
print(f"\u2705 Validation dataset formatted: {len(eval_dataset)} examples")

from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=1,
    max_steps=75,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="./llama_function_calling_finetune",
    report_to="none",
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=25,
    max_grad_norm=1.0,
)

print(f"\n\u2705 Training configuration ready!")
print(f"   - Batch size: {training_config.per_device_train_batch_size}")
print(f"   - Learning rate: {training_config.learning_rate}")
print(f"   - Epochs: {training_config.num_train_epochs}")
print(f"   - Training dataset: {len(dataset)}")
print(f"   - Eval dataset: {len(eval_dataset)}")
print(f"   - Eval every: {training_config.eval_steps} steps")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    args=training_config,
)

print("\u2705 SFTTrainer initialized successfully!")
print("\n\U0001f680 Ready to start training...")

🔄 Formatting training dataset with per-example tools...
✅ Training dataset formatted: 271 examples
Sample text length: 1394 characters
First 300 chars:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Cutting Knowledge Date: December 2023
Today Date: 01 Jun 2026

Generate search queries for knowledge retrieval.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following functions, please respond with a JS...

🔄 Formatting validation dataset for eval during training...
✅ Validation dataset formatted: 28 examples

✅ Training configuration ready!
   - Batch size: 2
   - Learning rate: 0.0001
   - Epochs: 3
   - Training dataset: 271
   - Eval dataset: 28
   - Eval every: 25 steps


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/271 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/28 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ SFTTrainer initialized successfully!

🚀 Ready to start training...


In [8]:
# ============================================================================
# COMPREHENSIVE EVALUATION FUNCTION (MANY METRICS)
# ============================================================================

import time
import numpy as np
from collections import defaultdict

try:
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    HAS_NLTK = True
except:
    HAS_NLTK = False
    print("\u26a0\ufe0f NLTK not available")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except:
    HAS_ROUGE = False
    print("\u26a0\ufe0f rouge-score not available")

try:
    import Levenshtein
    HAS_LEVENSHTEIN = True
except:
    HAS_LEVENSHTEIN = False
    print("\u26a0\ufe0f python-Levenshtein not available")

try:
    from sklearn.metrics import precision_recall_fscore_support
    HAS_SKLEARN = True
except:
    HAS_SKLEARN = False
    print("\u26a0\ufe0f scikit-learn not available")


def compute_bleu(reference, hypothesis):
    if not HAS_NLTK or not reference or not hypothesis:
        return 0.0
    try:
        ref_tokens = list(reference)
        hyp_tokens = list(hypothesis)
        smoothie = SmoothingFunction().method1
        return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothie)
    except:
        return 0.0

def compute_rouge_l(reference, hypothesis):
    if not HAS_ROUGE or not reference or not hypothesis:
        return 0.0
    try:
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
        scores = scorer.score(reference, hypothesis)
        return scores['rougeL'].fmeasure
    except:
        return 0.0

def compute_levenshtein_similarity(reference, hypothesis):
    if not HAS_LEVENSHTEIN or not reference or not hypothesis:
        return 0.0
    try:
        distance = Levenshtein.distance(reference, hypothesis)
        max_len = max(len(reference), len(hypothesis))
        return 1.0 - (distance / max_len) if max_len > 0 else 1.0
    except:
        return 0.0

def compute_cosine_similarity_bow(reference, hypothesis):
    if not reference or not hypothesis:
        return 0.0
    try:
        def get_ngrams(text, n=3):
            return [text[i:i+n] for i in range(len(text)-n+1)]
        ref_ngrams = get_ngrams(reference)
        hyp_ngrams = get_ngrams(hypothesis)
        vocab = set(ref_ngrams + hyp_ngrams)
        if not vocab:
            return 0.0
        ref_counts = defaultdict(int)
        hyp_counts = defaultdict(int)
        for ng in ref_ngrams:
            ref_counts[ng] += 1
        for ng in hyp_ngrams:
            hyp_counts[ng] += 1
        dot = sum(ref_counts[w] * hyp_counts[w] for w in vocab)
        norm_ref = sum(v**2 for v in ref_counts.values()) ** 0.5
        norm_hyp = sum(v**2 for v in hyp_counts.values()) ** 0.5
        if norm_ref == 0 or norm_hyp == 0:
            return 0.0
        return dot / (norm_ref * norm_hyp)
    except:
        return 0.0


def evaluate_function_calling(model, tokenizer, validation_examples, model_name="Model", max_examples=None, show_samples=3):
    """
    Comprehensive evaluation with nested JSON support, attention_mask fix,
    and tools passed during evaluation to match training format.

    Key fixes vs previous version:
    1. JSON extraction uses bracket-counting (handles nested JSON)
    2. attention_mask explicitly passed to model.generate()
    3. Tools from validation example passed to apply_chat_template
    4. Checks both 'parameters' and 'arguments' style responses
    5. Shows sample model outputs for debugging
    """
    import torch

    if max_examples is None:
        max_examples = len(validation_examples)
    else:
        max_examples = min(max_examples, len(validation_examples))

    metrics = {
        "total": max_examples,
        "valid_json": 0,
        "has_name_field": 0,
        "has_parameters_field": 0,
        "has_intent_field": 0,
        "has_reasoning_field": 0,
        "correct_intent": 0,
        "accuracy_by_intent": defaultdict(lambda: {"total": 0, "correct": 0}),
        "exact_match": 0,
        "bleu_scores": [],
        "rouge_l_scores": [],
        "levenshtein_scores": [],
        "cosine_scores": [],
        "latencies": [],
        "output_token_counts": [],
        "true_intents": [],
        "pred_intents": [],
        "error_count": 0,
        "errors": [],
        "sample_outputs": [],  # Store sample outputs for debugging
    }

    print(f"\n{'='*80}")
    print(f"\U0001f9ea EVALUATING {model_name}")
    print(f"{'='*80}")
    print(f"Evaluating on {max_examples} examples...\n")

    for idx in range(max_examples):
        example = validation_examples[idx]
        user_input = example["user_input"]
        system_prompt = example.get("system_prompt", "")
        expected_intent = example.get("expected_intent")
        expected_response_raw = example.get("assistant_response", "")
        example_tools = example.get("tools")  # Get tools for this example

        try:
            messages = [
                {
                    "role": "system",
                    "content": system_prompt if system_prompt else "You are a market research analyst. Classify user intent and respond appropriately."
                },
                {
                    "role": "user",
                    "content": user_input
                }
            ]

            # FIX: Pass tools to match training format
            if hasattr(tokenizer, "apply_chat_template"):
                try:
                    model_input = tokenizer.apply_chat_template(
                        messages,
                        tools=example_tools if example_tools else None,
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                except:
                    model_input = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=True,
                    )
            else:
                model_input = user_input

            model_inputs = tokenizer([model_input], return_tensors="pt")

            if torch.cuda.is_available():
                model_inputs = model_inputs.to('cuda')

            # FIX: Explicitly pass attention_mask
            start_time = time.time()
            with torch.no_grad():
                generated_ids = model.generate(
                    model_inputs.input_ids,
                    attention_mask=model_inputs.attention_mask,
                    max_new_tokens=300,
                    do_sample=False,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id,
                )
            end_time = time.time()
            metrics["latencies"].append(end_time - start_time)

            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
            metrics["output_token_counts"].append(len(output_ids))

            response = tokenizer.decode(output_ids, skip_special_tokens=True)

            # Store sample outputs for debugging
            if idx < show_samples:
                metrics["sample_outputs"].append({
                    "idx": idx,
                    "user_input": user_input[:80],
                    "expected": expected_response_raw[:150],
                    "generated": response[:300],
                    "expected_intent": expected_intent,
                })

            # Exact Match
            if response.strip() == expected_response_raw.strip():
                metrics["exact_match"] += 1

            # Text Similarity
            metrics["bleu_scores"].append(compute_bleu(expected_response_raw, response))
            metrics["rouge_l_scores"].append(compute_rouge_l(expected_response_raw, response))
            metrics["levenshtein_scores"].append(compute_levenshtein_similarity(expected_response_raw, response))
            metrics["cosine_scores"].append(compute_cosine_similarity_bow(expected_response_raw, response))

            # FIX: Use bracket-counting JSON extraction (handles nested JSON)
            json_data = extract_json_from_response(response)

            if json_data:
                metrics["valid_json"] += 1

                # Field Presence
                predicted_intent = None

                if "name" in json_data:
                    metrics["has_name_field"] += 1

                # Check 'parameters' style
                if "parameters" in json_data:
                    metrics["has_parameters_field"] += 1
                    params = json_data["parameters"]
                    if isinstance(params, dict):
                        if "intent" in params:
                            metrics["has_intent_field"] += 1
                            predicted_intent = params["intent"]
                        if "reasoning" in params:
                            metrics["has_reasoning_field"] += 1

                # Check 'arguments' style (training data uses this format)
                if "arguments" in json_data:
                    metrics["has_parameters_field"] += (1 if "parameters" not in json_data else 0)
                    args = json_data["arguments"]
                    if isinstance(args, dict):
                        if "intent" in args and not predicted_intent:
                            metrics["has_intent_field"] += 1
                            predicted_intent = args["intent"]
                        if "reasoning" in args:
                            metrics["has_reasoning_field"] += 1

                # Intent Accuracy
                if expected_intent:
                    metrics["true_intents"].append(expected_intent)
                    metrics["pred_intents"].append(predicted_intent if predicted_intent else "__NONE__")
                    metrics["accuracy_by_intent"][expected_intent]["total"] += 1
                    if predicted_intent and predicted_intent == expected_intent:
                        metrics["correct_intent"] += 1
                        metrics["accuracy_by_intent"][expected_intent]["correct"] += 1
            else:
                # No valid JSON found
                if expected_intent:
                    metrics["true_intents"].append(expected_intent)
                    metrics["pred_intents"].append("__NONE__")
                    metrics["accuracy_by_intent"][expected_intent]["total"] += 1

        except Exception as e:
            metrics["error_count"] += 1
            metrics["errors"].append({
                "example_idx": idx,
                "user_input": user_input[:50],
                "error": str(e)
            })

        if (idx + 1) % max(1, max_examples // 10) == 0:
            print(f"  Processed {idx + 1}/{max_examples} examples...")

    # ============================================================================
    # COMPUTE AGGREGATED METRICS
    # ============================================================================
    total = metrics["total"]

    metrics["valid_json_rate"] = metrics["valid_json"] / total * 100 if total > 0 else 0
    metrics["name_field_rate"] = metrics["has_name_field"] / total * 100 if total > 0 else 0
    metrics["parameters_field_rate"] = metrics["has_parameters_field"] / total * 100 if total > 0 else 0
    metrics["intent_field_rate"] = metrics["has_intent_field"] / total * 100 if total > 0 else 0
    metrics["reasoning_field_rate"] = metrics["has_reasoning_field"] / total * 100 if total > 0 else 0
    metrics["overall_accuracy"] = metrics["valid_json"] / total * 100 if total > 0 else 0
    metrics["intent_accuracy"] = metrics["correct_intent"] / total * 100 if total > 0 else 0
    metrics["exact_match_rate"] = metrics["exact_match"] / total * 100 if total > 0 else 0
    metrics["error_rate"] = metrics["error_count"] / total * 100 if total > 0 else 0

    metrics["avg_bleu"] = np.mean(metrics["bleu_scores"]) if metrics["bleu_scores"] else 0
    metrics["avg_rouge_l"] = np.mean(metrics["rouge_l_scores"]) if metrics["rouge_l_scores"] else 0
    metrics["avg_levenshtein"] = np.mean(metrics["levenshtein_scores"]) if metrics["levenshtein_scores"] else 0
    metrics["avg_cosine"] = np.mean(metrics["cosine_scores"]) if metrics["cosine_scores"] else 0
    metrics["avg_latency"] = np.mean(metrics["latencies"]) if metrics["latencies"] else 0
    metrics["avg_output_tokens"] = np.mean(metrics["output_token_counts"]) if metrics["output_token_counts"] else 0

    if HAS_SKLEARN and metrics["true_intents"]:
        labels = sorted(set(metrics["true_intents"] + metrics["pred_intents"]))
        precision, recall, f1, support = precision_recall_fscore_support(
            metrics["true_intents"], metrics["pred_intents"], labels=labels, average=None, zero_division=0
        )
        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
            metrics["true_intents"], metrics["pred_intents"], average="macro", zero_division=0
        )
        weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
            metrics["true_intents"], metrics["pred_intents"], average="weighted", zero_division=0
        )
        metrics["per_intent_prf"] = {}
        for i, label in enumerate(labels):
            metrics["per_intent_prf"][label] = {
                "precision": precision[i], "recall": recall[i],
                "f1": f1[i], "support": int(support[i]),
            }
        metrics["macro_precision"] = macro_p
        metrics["macro_recall"] = macro_r
        metrics["macro_f1"] = macro_f1
        metrics["weighted_precision"] = weighted_p
        metrics["weighted_recall"] = weighted_r
        metrics["weighted_f1"] = weighted_f1

    # ============================================================================
    # PRINT RESULTS
    # ============================================================================

    # Show sample outputs first (debugging)
    if metrics["sample_outputs"]:
        print(f"\n{'='*80}")
        print(f"\U0001f50d SAMPLE MODEL OUTPUTS (first {len(metrics['sample_outputs'])} examples)")
        print(f"{'='*80}")
        for sample in metrics["sample_outputs"]:
            print(f"\n--- Example {sample['idx']+1} ---")
            print(f"   User:     {sample['user_input']}")
            print(f"   Expected: {sample['expected'][:120]}...")
            print(f"   Got:      {sample['generated'][:200]}")
            # Check if JSON was extracted
            extracted = extract_json_from_response(sample['generated'])
            if extracted:
                print(f"   JSON OK:  {json.dumps(extracted, ensure_ascii=False)[:150]}")
            else:
                print(f"   JSON:     \u274c Not found")

    print(f"\n{'='*80}")
    print(f"\U0001f4ca EVALUATION RESULTS FOR {model_name}")
    print(f"{'='*80}")

    print(f"\n\U0001f4c8 Structure Metrics:")
    print(f"   \u2022 Total examples:         {total}")
    print(f"   \u2022 Valid JSON:              {metrics['valid_json']}/{total} ({metrics['valid_json_rate']:.1f}%)")
    print(f"   \u2022 Has 'name' field:        {metrics['has_name_field']}/{total} ({metrics['name_field_rate']:.1f}%)")
    print(f"   \u2022 Has 'params/args' field:  {metrics['has_parameters_field']}/{total} ({metrics['parameters_field_rate']:.1f}%)")
    print(f"   \u2022 Has 'intent' field:      {metrics['has_intent_field']}/{total} ({metrics['intent_field_rate']:.1f}%)")
    print(f"   \u2022 Has 'reasoning' field:   {metrics['has_reasoning_field']}/{total} ({metrics['reasoning_field_rate']:.1f}%)")

    print(f"\n\U0001f3af Intent Classification:")
    print(f"   \u2022 Correct intent:          {metrics['correct_intent']}/{total} ({metrics['intent_accuracy']:.1f}%)")
    if 'macro_f1' in metrics:
        print(f"   \u2022 Macro Precision:         {metrics['macro_precision']:.4f}")
        print(f"   \u2022 Macro Recall:            {metrics['macro_recall']:.4f}")
        print(f"   \u2022 Macro F1:                {metrics['macro_f1']:.4f}")
        print(f"   \u2022 Weighted Precision:      {metrics['weighted_precision']:.4f}")
        print(f"   \u2022 Weighted Recall:         {metrics['weighted_recall']:.4f}")
        print(f"   \u2022 Weighted F1:             {metrics['weighted_f1']:.4f}")

    print(f"\n\U0001f4dd Response Quality:")
    print(f"   \u2022 Exact Match Rate:        {metrics['exact_match']}/{total} ({metrics['exact_match_rate']:.1f}%)")
    print(f"   \u2022 Avg BLEU Score:          {metrics['avg_bleu']:.4f}")
    print(f"   \u2022 Avg ROUGE-L Score:       {metrics['avg_rouge_l']:.4f}")
    print(f"   \u2022 Avg Levenshtein Sim:     {metrics['avg_levenshtein']:.4f}")
    print(f"   \u2022 Avg Cosine Sim (BoW):    {metrics['avg_cosine']:.4f}")

    print(f"\n\u26a1 Performance:")
    print(f"   \u2022 Avg Latency:             {metrics['avg_latency']:.3f}s")
    print(f"   \u2022 Avg Output Tokens:       {metrics['avg_output_tokens']:.1f}")
    print(f"   \u2022 Error Rate:              {metrics['error_count']}/{total} ({metrics['error_rate']:.1f}%)")

    if metrics.get("accuracy_by_intent"):
        print(f"\n\U0001f3af Per-Intent Accuracy:")
        for intent, stats in sorted(metrics["accuracy_by_intent"].items()):
            acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            print(f"   \u2022 {intent}: {stats['correct']}/{stats['total']} ({acc:.1f}%)")

    if metrics.get("per_intent_prf"):
        print(f"\n\U0001f4cb Per-Intent Precision / Recall / F1:")
        print(f"   {'Intent':<20} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Support':>8}")
        print(f"   {'\u2500'*20} {'\u2500'*8} {'\u2500'*8} {'\u2500'*8} {'\u2500'*8}")
        for intent, prf in sorted(metrics["per_intent_prf"].items()):
            if intent != "__NONE__":
                print(f"   {intent:<20} {prf['precision']:>8.4f} {prf['recall']:>8.4f} {prf['f1']:>8.4f} {prf['support']:>8}")

    print(f"\n{'='*80}\n")
    return metrics


print("\u2705 Comprehensive evaluation function defined successfully!")
print("   Fixes applied: nested JSON parser, attention_mask, tools in eval")

✅ Comprehensive evaluation function defined successfully!
   Fixes applied: nested JSON parser, attention_mask, tools in eval


In [9]:
# ============================================================================
# EVALUATION BEFORE TRAINING (PRE-TRAINING BASELINE)
# ============================================================================

print("\n" + "="*80)
print("\U0001f50d PRE-TRAINING EVALUATION (BASELINE)")
print("="*80)
print("Evaluating base model before fine-tuning...")
print(f"Using {len(validation_examples)} validation examples (10% of training data)")
print("\nNOTE: Base model was NOT trained for function calling,")
print("so low scores are EXPECTED. This is the baseline for comparison.\n")

# Prepare model for inference
FastLanguageModel.for_inference(model)

# Run comprehensive evaluation on validation set (show 5 sample outputs)
pre_training_metrics = evaluate_function_calling(
    model,
    tokenizer,
    validation_examples,
    model_name="Base Model (Before Training)",
    max_examples=len(validation_examples),
    show_samples=5
)

print("\U0001f4be Pre-training metrics stored for comparison")


🔍 PRE-TRAINING EVALUATION (BASELINE)
Evaluating base model before fine-tuning...
Using 28 validation examples (10% of training data)

NOTE: Base model was NOT trained for function calling,
so low scores are EXPECTED. This is the baseline for comparison.


🧪 EVALUATING Base Model (Before Training)
Evaluating on 28 examples...



Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

  Processed 2/28 examples...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed 4/28 examples...


...

  Processed 24/28 examples...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed 26/28 examples...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed 28/28 examples...

🔍 SAMPLE MODEL OUTPUTS (first 5 examples)

--- Example 1 ---
   User:     Tổng hợp dữ liệu nghiên cứu thị trường xe điện.
   Expected: {"name":"synthesize_overview","arguments":{"tong_quan_thi_truong":"Thị trường xe điện tăng trưởng nhanh nhờ chính sách h...
   Got:      {"name": "synthesize_overview", "parameters": {"tong_quan_thi_truong": "Tổng hợp dữ liệu nghiên cứu thị trường xe điện", "tro_ly": "[\"Tăng trưởng nhanh nhất trong 5 năm qua, với 1,2 triệu xe điện đượ
   JSON OK:  {"name": "synthesize_overview", "parameters": {"tong_quan_thi_truong": "Tổng hợp dữ liệu nghiên cứu thị trường xe điện", "tro_ly": "[\"Tăng trưởng nha

--- Example 2 ---
   User:     Lập campaign Discord 7 ngày cho game mới.
   Expected: {"name":"create_campaign_plan","arguments":{"duration_days":7,"posting_frequency":"3 times daily","schedule":[{"day":1,"...
   Got:      {"name": "create_campaign_plan", "parameters": {"duration_days": "7", "posting_frequency": "2-3 times daily"

In [10]:
# ============================================================================
# START TRAINING WITH TOOL SUPPORT + EVAL ON VALIDATION
# ============================================================================

print("\n" + "="*70)
print("\U0001f680 STARTING FINE-TUNING WITH TOOL-AWARE FUNCTION CALLING")
print("="*70 + "\n")
print(f"Training on {len(dataset)} examples, evaluating on {len(eval_dataset)} examples")
print(f"Evaluation every {training_config.eval_steps} steps\n")

model.train()

train_result = trainer.train()

print("\n" + "="*70)
print("\u2705 TRAINING COMPLETED!")
print("="*70)

print(f"\n\U0001f4ca Training Statistics:")
print(f"   \u2022 Total steps:       {train_result.global_step}")
print(f"   \u2022 Training loss:     {train_result.training_loss:.4f}")
if hasattr(train_result, 'metrics'):
    for k, v in train_result.metrics.items():
        if isinstance(v, float):
            print(f"   \u2022 {k}: {v:.4f}")
        else:
            print(f"   \u2022 {k}: {v}")

output_dir = "./llama_function_calling_finetune_final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n\U0001f4be Model saved to: {output_dir}")


🚀 STARTING FINE-TUNING WITH TOOL-AWARE FUNCTION CALLING

Training on 271 examples, evaluating on 28 examples
Evaluation every 25 steps



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 271 | Num Epochs = 2 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,0.625457,0.601368
50,0.488096,0.416905
75,0.348624,0.385651



✅ TRAINING COMPLETED!

📊 Training Statistics:
   • Total steps:       75
   • Training loss:     0.8100
   • train_runtime: 238.2859
   • train_samples_per_second: 1.2590
   • train_steps_per_second: 0.3150
   • total_flos: 1631291682877440.0000
   • train_loss: 0.8100
   • epoch: 1.1029


Unsloth: Restored added_tokens_decoder metadata in ./llama_function_calling_finetune_final/tokenizer_config.json.



💾 Model saved to: ./llama_function_calling_finetune_final


In [11]:
# ============================================================================
# EVALUATION AFTER TRAINING (POST-TRAINING)
# ============================================================================

print("\n" + "="*80)
print("\U0001f50d POST-TRAINING EVALUATION")
print("="*80)
print("Evaluating fine-tuned model...")
print(f"Using {len(validation_examples)} validation examples (same 10% split)\n")

FastLanguageModel.for_inference(model)

post_training_metrics = evaluate_function_calling(
    model,
    tokenizer,
    validation_examples,
    model_name="Fine-Tuned Model (After Training)",
    max_examples=len(validation_examples),
    show_samples=5
)

print("\U0001f4be Post-training metrics stored for comparison")

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 POST-TRAINING EVALUATION
Evaluating fine-tuned model...
Using 28 validation examples (same 10% split)


🧪 EVALUATING Fine-Tuned Model (After Training)
Evaluating on 28 examples...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=

  Processed 2/28 examples...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Processed 24/28 examples...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed 26/28 examples...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed 28/28 examples...

🔍 SAMPLE MODEL OUTPUTS (first 5 examples)

--- Example 1 ---
   User:     Tổng hợp dữ liệu nghiên cứu thị trường xe điện.
   Expected: {"name":"synthesize_overview","arguments":{"tong_quan_thi_truong":"Thị trường xe điện tăng trưởng nhanh nhờ chính sách h...
   Got:      {"name":"synthesize_overview","arguments":{"tong_quan_thi_truong":"Xe điện đang tăng trưởng mạnh, nhưng vẫn còn nhiều thách thức","tro_ly":["Tăng trưởng nhanh nhưng vẫn còn hạn chế","Nhu cầu tăng mạnh
   JSON OK:  {"name": "synthesize_overview", "arguments": {"tong_quan_thi_truong": "Xe điện đang tăng trưởng mạnh, nhưng vẫn còn nhiều thách thức", "tro_ly": ["Tăn

--- Example 2 ---
   User:     Lập campaign Discord 7 ngày cho game mới.
   Expected: {"name":"create_campaign_plan","arguments":{"duration_days":7,"posting_frequency":"3 times daily","schedule":[{"day":1,"...
   Got:      {"name":"create_campaign_plan","arguments":{"duration_days":7,"posting_frequency":"3-4 times daily","schedul

In [12]:
# ============================================================================
# COMPREHENSIVE COMPARISON: BEFORE vs AFTER TRAINING
# ============================================================================

import json

def compare_metrics(pre_metrics, post_metrics):
    print("\n" + "="*90)
    print("\U0001f4ca COMPREHENSIVE TRAINING IMPACT ANALYSIS: BEFORE vs AFTER")
    print("="*90)

    total = pre_metrics["total"]

    # TABLE 1: Structure & Accuracy
    print(f"\n{'\u2500'*90}")
    print(f"  \U0001f4cb TABLE 1: STRUCTURE & ACCURACY METRICS")
    print(f"{'\u2500'*90}")
    print(f"  {'Metric':<35} {'Before':>12} {'After':>12} {'Change':>12} {'Verdict':>10}")
    print(f"  {'\u2500'*35} {'\u2500'*12} {'\u2500'*12} {'\u2500'*12} {'\u2500'*10}")

    count_metrics = [
        ("Valid JSON Generation", "valid_json"),
        ("'name' Field Presence", "has_name_field"),
        ("'params/args' Field Presence", "has_parameters_field"),
        ("'intent' Field Presence", "has_intent_field"),
        ("'reasoning' Field Presence", "has_reasoning_field"),
        ("Correct Intent", "correct_intent"),
        ("Exact Match", "exact_match"),
    ]

    for name, key in count_metrics:
        pre_val = pre_metrics.get(key, 0)
        post_val = post_metrics.get(key, 0)
        pre_pct = pre_val / total * 100 if total > 0 else 0
        post_pct = post_val / total * 100 if total > 0 else 0
        diff = post_pct - pre_pct
        diff_str = f"+{diff:.1f}%" if diff >= 0 else f"{diff:.1f}%"
        verdict = "\u2705 Better" if diff > 0 else ("\u2796 Same" if diff == 0 else "\u274c Worse")
        print(f"  {name:<35} {pre_pct:>10.1f}% {post_pct:>10.1f}% {diff_str:>12} {verdict:>10}")

    # TABLE 2: Text Similarity
    print(f"\n{'\u2500'*90}")
    print(f"  \U0001f4dd TABLE 2: TEXT SIMILARITY METRICS")
    print(f"{'\u2500'*90}")
    print(f"  {'Metric':<35} {'Before':>12} {'After':>12} {'Change':>12} {'Verdict':>10}")
    print(f"  {'\u2500'*35} {'\u2500'*12} {'\u2500'*12} {'\u2500'*12} {'\u2500'*10}")

    for name, key in [("Avg BLEU Score", "avg_bleu"), ("Avg ROUGE-L Score", "avg_rouge_l"),
                      ("Avg Levenshtein Similarity", "avg_levenshtein"), ("Avg Cosine Similarity (BoW)", "avg_cosine")]:
        pre_val = pre_metrics.get(key, 0)
        post_val = post_metrics.get(key, 0)
        diff = post_val - pre_val
        diff_str = f"+{diff:.4f}" if diff >= 0 else f"{diff:.4f}"
        verdict = "\u2705 Better" if diff > 0.001 else ("\u2796 Same" if abs(diff) <= 0.001 else "\u274c Worse")
        print(f"  {name:<35} {pre_val:>12.4f} {post_val:>12.4f} {diff_str:>12} {verdict:>10}")

    # TABLE 3: Classification P/R/F1
    print(f"\n{'\u2500'*90}")
    print(f"  \U0001f3af TABLE 3: CLASSIFICATION METRICS (Precision / Recall / F1)")
    print(f"{'\u2500'*90}")
    print(f"  {'Metric':<35} {'Before':>12} {'After':>12} {'Change':>12} {'Verdict':>10}")
    print(f"  {'\u2500'*35} {'\u2500'*12} {'\u2500'*12} {'\u2500'*12} {'\u2500'*10}")

    for name, key in [("Macro Precision", "macro_precision"), ("Macro Recall", "macro_recall"),
                      ("Macro F1", "macro_f1"), ("Weighted Precision", "weighted_precision"),
                      ("Weighted Recall", "weighted_recall"), ("Weighted F1", "weighted_f1")]:
        pre_val = pre_metrics.get(key, 0)
        post_val = post_metrics.get(key, 0)
        diff = post_val - pre_val
        diff_str = f"+{diff:.4f}" if diff >= 0 else f"{diff:.4f}"
        verdict = "\u2705 Better" if diff > 0.001 else ("\u2796 Same" if abs(diff) <= 0.001 else "\u274c Worse")
        print(f"  {name:<35} {pre_val:>12.4f} {post_val:>12.4f} {diff_str:>12} {verdict:>10}")

    # TABLE 4: Performance
    print(f"\n{'\u2500'*90}")
    print(f"  \u26a1 TABLE 4: PERFORMANCE & EFFICIENCY")
    print(f"{'\u2500'*90}")
    print(f"  {'Metric':<35} {'Before':>12} {'After':>12} {'Change':>12} {'Verdict':>10}")
    print(f"  {'\u2500'*35} {'\u2500'*12} {'\u2500'*12} {'\u2500'*12} {'\u2500'*10}")

    pre_lat = pre_metrics.get("avg_latency", 0)
    post_lat = post_metrics.get("avg_latency", 0)
    lat_diff = post_lat - pre_lat
    print(f"  {'Avg Latency':<35} {pre_lat:>10.3f}s {post_lat:>10.3f}s {'+' if lat_diff>=0 else ''}{lat_diff:.3f}s")

    pre_tok = pre_metrics.get("avg_output_tokens", 0)
    post_tok = post_metrics.get("avg_output_tokens", 0)
    print(f"  {'Avg Output Tokens':<35} {pre_tok:>12.1f} {post_tok:>12.1f} {'+' if post_tok-pre_tok>=0 else ''}{post_tok-pre_tok:.1f}")

    pre_err = pre_metrics.get("error_count", 0)
    post_err = post_metrics.get("error_count", 0)
    print(f"  {'Errors':<35} {pre_err:>12} {post_err:>12} {'+' if post_err-pre_err>=0 else ''}{post_err-pre_err}")

    # TABLE 5: Per-Intent Accuracy
    all_intents = set(pre_metrics["accuracy_by_intent"].keys()) | set(post_metrics["accuracy_by_intent"].keys())
    if all_intents:
        print(f"\n{'\u2500'*90}")
        print(f"  \U0001f3f7\ufe0f  TABLE 5: PER-INTENT ACCURACY COMPARISON")
        print(f"{'\u2500'*90}")
        print(f"  {'Intent':<20} {'Before':>15} {'After':>15} {'Change':>12}")
        print(f"  {'\u2500'*20} {'\u2500'*15} {'\u2500'*15} {'\u2500'*12}")
        for intent in sorted(all_intents):
            pre_stats = pre_metrics["accuracy_by_intent"].get(intent, {"total": 0, "correct": 0})
            post_stats = post_metrics["accuracy_by_intent"].get(intent, {"total": 0, "correct": 0})
            pre_acc = (pre_stats["correct"] / pre_stats["total"] * 100) if pre_stats["total"] > 0 else 0
            post_acc = (post_stats["correct"] / post_stats["total"] * 100) if post_stats["total"] > 0 else 0
            diff = post_acc - pre_acc
            print(f"  {intent:<20} {pre_acc:>5.1f}% ({pre_stats['correct']}/{pre_stats['total']}) {post_acc:>5.1f}% ({post_stats['correct']}/{post_stats['total']}) {'+' if diff>=0 else ''}{diff:.1f}%")

    # SUMMARY
    print(f"\n{'='*90}")
    print(f"  \U0001f4c8 OVERALL SUMMARY")
    print(f"{'='*90}")
    summary = [
        ("Valid JSON Rate", pre_metrics.get('valid_json_rate', 0), post_metrics.get('valid_json_rate', 0), "%"),
        ("Intent Accuracy", pre_metrics.get('intent_accuracy', 0), post_metrics.get('intent_accuracy', 0), "%"),
        ("Exact Match Rate", pre_metrics.get('exact_match_rate', 0), post_metrics.get('exact_match_rate', 0), "%"),
        ("BLEU Score", pre_metrics.get('avg_bleu', 0)*100, post_metrics.get('avg_bleu', 0)*100, "%"),
        ("ROUGE-L Score", pre_metrics.get('avg_rouge_l', 0)*100, post_metrics.get('avg_rouge_l', 0)*100, "%"),
        ("Macro F1", pre_metrics.get('macro_f1', 0)*100, post_metrics.get('macro_f1', 0)*100, "%"),
    ]
    total_imp = 0
    for name, pre_val, post_val, unit in summary:
        diff = post_val - pre_val
        total_imp += diff
        arrow = "\U0001f4c8" if diff > 0 else ("\U0001f4c9" if diff < 0 else "\u27a1\ufe0f")
        print(f"   {arrow} {name}: {pre_val:.1f}{unit} \u2192 {post_val:.1f}{unit} (\u0394 {'+' if diff>=0 else ''}{diff:.1f}{unit})")
    print(f"\n   \U0001f3c6 Average improvement: {'+' if total_imp/len(summary)>=0 else ''}{total_imp/len(summary):.2f}%")
    print(f"{'='*90}\n")


if 'pre_training_metrics' in locals() and 'post_training_metrics' in locals():
    compare_metrics(pre_training_metrics, post_training_metrics)

    def safe_serialize(obj):
        if isinstance(obj, dict):
            return {k: safe_serialize(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [safe_serialize(v) for v in obj]
        elif isinstance(obj, (int, float, str, bool, type(None))):
            return obj
        elif hasattr(obj, 'item'):
            return obj.item()
        else:
            return str(obj)

    exclude_keys = {"errors", "bleu_scores", "rouge_l_scores", "levenshtein_scores",
                    "cosine_scores", "latencies", "output_token_counts", "true_intents",
                    "pred_intents", "sample_outputs"}

    comparison_results = {
        "data_split": {
            "total_examples": len(all_examples),
            "training_examples": len(training_examples),
            "validation_examples": len(validation_examples),
            "split_ratio": "90/10",
            "split_method": "stratified by intent",
        },
        "pre_training": safe_serialize({k: v for k, v in pre_training_metrics.items() if k not in exclude_keys}),
        "post_training": safe_serialize({k: v for k, v in post_training_metrics.items() if k not in exclude_keys}),
        "improvement": {
            "valid_json_gain": post_training_metrics.get("valid_json_rate", 0) - pre_training_metrics.get("valid_json_rate", 0),
            "intent_accuracy_gain": post_training_metrics.get("intent_accuracy", 0) - pre_training_metrics.get("intent_accuracy", 0),
            "exact_match_gain": post_training_metrics.get("exact_match_rate", 0) - pre_training_metrics.get("exact_match_rate", 0),
            "bleu_gain": post_training_metrics.get("avg_bleu", 0) - pre_training_metrics.get("avg_bleu", 0),
            "rouge_l_gain": post_training_metrics.get("avg_rouge_l", 0) - pre_training_metrics.get("avg_rouge_l", 0),
            "macro_f1_gain": post_training_metrics.get("macro_f1", 0) - pre_training_metrics.get("macro_f1", 0),
            "weighted_f1_gain": post_training_metrics.get("weighted_f1", 0) - pre_training_metrics.get("weighted_f1", 0),
        }
    }

    comparison_file = "/content/drive/MyDrive/Colab Notebooks/AI/do an/evaluation_comparison.json"
    with open(comparison_file, "w", encoding="utf-8") as f:
        json.dump(comparison_results, f, indent=2, ensure_ascii=False)
    print(f"\U0001f4be Comparison saved to: {comparison_file}")
else:
    print("\u26a0\ufe0f Both pre-training and post-training metrics needed for comparison")


📊 COMPREHENSIVE TRAINING IMPACT ANALYSIS: BEFORE vs AFTER

──────────────────────────────────────────────────────────────────────────────────────────
  📋 TABLE 1: STRUCTURE & ACCURACY METRICS
──────────────────────────────────────────────────────────────────────────────────────────
  Metric                                    Before        After       Change    Verdict
  ─────────────────────────────────── ──────────── ──────────── ──────────── ──────────
  Valid JSON Generation                     67.9%      100.0%       +32.1%   ✅ Better
  'name' Field Presence                     67.9%      100.0%       +32.1%   ✅ Better
  'params/args' Field Presence              67.9%      100.0%       +32.1%   ✅ Better
  'intent' Field Presence                   35.7%       53.6%       +17.9%   ✅ Better
  'reasoning' Field Presence                10.7%        0.0%       -10.7%    ❌ Worse
  Correct Intent                            17.9%       42.9%       +25.0%   ✅ Better
  Exact Match           

In [13]:
# ============================================================================
# PUSH FINE-TUNED MODEL TO HUGGING FACE HUB
# ============================================================================

from huggingface_hub import login, HfApi

print("\n" + "="*70)
print("\U0001f680 PUSHING MODEL TO HUGGING FACE HUB")
print("="*70 + "\n")

print("\U0001f4dd Logging into Hugging Face...")
print("   (You need a write access token from https://huggingface.co/settings/tokens)")

try:
    login("")
    print("\u2705 Logged in successfully!")
except Exception as e:
    print(f"\u26a0\ufe0f Login failed: {e}")
    print("   Skipping Hugging Face push...")
    exit()

repo_id = "justrandomname/function-calling"
print(f"\n\U0001f4e4 Pushing model to {repo_id}...")

try:
    model.push_to_hub(
        repo_id=repo_id, token=None, private=False,
        commit_message="Fine-tuned Llama-3.2-3B for function calling with tool support"
    )
    print(f"\u2705 Model pushed successfully!")

    print(f"\U0001f4e4 Pushing tokenizer...")
    tokenizer.push_to_hub(repo_id=repo_id, token=None, private=False)
    print(f"\u2705 Tokenizer pushed successfully!")

    print(f"\U0001f4e4 Pushing training config...")
    training_config.push_to_hub(repo_id=repo_id, token=None)
    print(f"\u2705 Training config pushed successfully!")

    print("\n" + "="*70)
    print(f"\U0001f389 SUCCESS! Model available at:")
    print(f"   https://huggingface.co/{repo_id}")
    print("="*70)
except Exception as e:
    print(f"\u274c Push failed: {e}")
    print("   Make sure your token has write access to the repo")


🚀 PUSHING MODEL TO HUGGING FACE HUB

📝 Logging into Hugging Face...
   (You need a write access token from https://huggingface.co/settings/tokens)
⚠️ Login failed: Illegal header value b'Bearer '
   Skipping Hugging Face push...

📤 Pushing model to justrandomname/function-calling...
❌ Push failed: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a1d46f8-0f9c0f71736b406438fd7533;d45c7600-5ed0-48e0-850b-f6f48bfe5adb)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.
   Make sure your token has write access to the repo


In [ ]:
!cp -r "./llama_function_calling_finetune_final" "/content/drive/MyDrive/Colab Notebooks/AI/do an/"